### Catalog uniformization

The following code may need to be edited to work for each specific catalog.

We cut the full catalog to subsection of confident galaxies

We rename columns with standardized schema (id, ra, dec, z, z unc, and any unique values that we want to keep for a given catalog. If possible save the magnitude of the galaxy as host_mag)

- z_unc saves the 1 sigma uncertainty of the redshift.

- If possible we save ra/dec uncertainties, but if they only exist as global values they are recorded in catalogs_catalog.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gzip
from astropy.table import Table, hstack, vstack
from astropy.io import fits

In [ ]:
# visualize current catalog redshift coverage.
# fixme plot galaxy luminosity/mass distribution, redshift vs luminosity param space
from thor.utils.filter_functions import plot_redshift_histogram
plot_redshift_histogram()

This code processes the Astrodeep catalogs from JWST

Note on Astrodeep: No per-source astrometric errors are provided in the ASTRODEEP-JWST catalog (Merlin et al. 2024). Images are aligned to Gaia-DR3; we adopt a conservative positional uncertainty of 0.5 arcsec for crossmatching.

In [ ]:
# we load the full downloaded catalog

with fits.open('../../data/catalogs/raw_catalogs/ASTRODEEP-JWST_photoz/ABELL2744_photoz.fits') as hdul:
    hdul.info()

t = Table.read('../../data/catalogs/raw_catalogs/ASTRODEEP-JWST_photoz/ABELL2744_photoz.fits')                                                                                                 
t[0:5]

In [ ]:
# inspect catalog contents, ie histogram redshift values

import matplotlib.pyplot as plt

plot = plt.hist(t['zspec'], bins=50, range=(0,6))

In [ ]:
# here are our cuts for the astrodeep catalogs (http://www.astrodeep.eu/astrodeep-jwst-catalogs/)
# see Merlin 2024 for description of data, flags etc (https://arxiv.org/abs/2409.00169)

galaxies = t[
    (t['flag'] < 100000) # spurious sources
]

print(f'selected {len(galaxies)} from {len(t)} total objects')

# and then here is where we standardize column names from the astrodeep names
# we dont cut any columns as astrodeep is already very concise.

cut_catalog = galaxies.copy()
cut_catalog.rename_column('ID', 'id')
cut_catalog.rename_column('RA', 'ra')
cut_catalog.rename_column('DEC', 'dec')
cut_catalog['z'] = np.where(cut_catalog['zspec'] != -99.0, cut_catalog['zspec'], cut_catalog['zphot']) # use zspec where possible  

# z uncertainty                         
# spec-z: use global assumption σ_z/(1+z) = 0.001 (typical MOS precision;
#         paper validates spec-z quality via flag filtering but gives no explicit value)                                                                                    
# photo-z: per-object std of the 4 EAzY estimates, floored at the global NMAD
#          NMAD = 0.031 from Merlin+2024 (Table in paper)                                                                                                                   
                                                                                                                                                                            
_SPECZ_SIGMA_FRAC = 0.001   # σ_z/(1+z) for spec-z                                                                                                                          
_PHOTOZ_NMAD      = 0.031   # global photo-z NMAD from paper                                                                                                                
                                                                                                                                                                            
_photoz_cols = ['EAzY_eazy_v13', 'EAzY_Larson', 'EAzY_LarsonLyaRed', 'zphot']                                                                                               
_photoz_stack = np.column_stack([cut_catalog[c] for c in _photoz_cols]).astype(float)                                                                                       
_photoz_per_obj_std = np.std(_photoz_stack, axis=1)                                                                                                                         
_photoz_nmad_floor  = _PHOTOZ_NMAD * (1 + cut_catalog['z'])                                                                                                                 
                                            
_has_specz = cut_catalog['zspec'] != -99.0                                                                                                                                  
cut_catalog['z_unc'] = np.where(                                                                                                                                          
    _has_specz,                                                                                                                                                             
    _SPECZ_SIGMA_FRAC * (1 + cut_catalog['z']),                                                                                                                           
    np.maximum(_photoz_per_obj_std, _photoz_nmad_floor),                                                                                                                    
) 

In [ ]:
# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

# Sky coverage (assuming contiguous rectangular field)
ra_min, ra_max = cut_catalog['ra'].min(), cut_catalog['ra'].max()
dec_min, dec_max = cut_catalog['dec'].min(), cut_catalog['dec'].max()
dec_mean_rad = np.radians((dec_min + dec_max) / 2)
area_sqdeg = (ra_max - ra_min) * np.cos(dec_mean_rad) * (dec_max - dec_min)
print(f'Approximate sky coverage: {area_sqdeg:.2f} deg^2')

# object stats
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

# finally we save the cut catalog to a fits file
cut_catalog.write('../../data/catalogs/ASTRODEEP_ABELL2744_cut.fits', overwrite=False) 

In [ ]:
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(
    alerts=None,
    plot_catalogs=True,
    plot_ddf=False,
    catalog_path="../../data/catalogs",
    title="Catalog Coverage Skymap",
)

This code processes the Cosmos2025 catalog from JWST

In [ ]:
# we load the full downloaded catalog. 
# combine 1: Photometry (PSF-homogenized Aperture and SE++ Model-Based) and 2: LePhare Photometric Redshifts and Physical Parameters

with fits.open('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_photom_primary.fits') as hdul:
    hdul.info()

with fits.open('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_lephare.fits') as hdul:
    hdul.info()
                                                                                                                                                        
t_photom = Table.read('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_photom_primary.fits')                                                            
t_lephare = Table.read('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_lephare.fits')                                                                
                                                                                                                                                        
# Both tables have 784016 rows in the same order — row-matched, so hstack directly                                                                       
t = hstack([t_photom, t_lephare])                                                                                                                      
t[0:5]                                                                                                                                                   
        

In [ ]:
# inspect catalog contents, ie histogram redshift values

import matplotlib.pyplot as plt

plot = plt.hist(t['zfinal'], bins=50, range=(0,6))

In [ ]:
# values to select on are specific to the catalog                                                                                                        
                                                                                                                                                        
# cuts for COSMOSWeb                                                                                  
# lephare type: 0=galaxy, 1=star, 2=AGN                                                                                                                  
galaxies = t[                                                                                                                                            
    (t['type'] == 0) &          # LePhare classified as galaxy                                                                                           
    (t['zfinal'] > 0) &         # has a valid redshift                                                                                                   
    (t['flag_star'] == False) &  # not a morphological star (SE++ flag)                                                                                  
    (t['flag_star_hsc'] == 0) &  # not a star per HSC (0=galaxy, 1=star)                                                                                 
    (t['flag_blend'] == False) & # not a blended source                                                                                                  
    (t['warn_flag'] == 0)        # no photometry/fit warnings                                                                                            
]                                                                                                                                                        
                                                                                                                                                        
print(f'selected {len(galaxies)} from {len(t)} total objects')                                                                                           
                                                                                                                                                        
# standardize column names                                                                                                                               
cut_catalog = galaxies['id', 'ra', 'dec', 'zfinal', 'zpdf_l68', 'zpdf_u68',                                                                              
                          'chi2_best', 'type', 'flag_chandra',                                                                                           
                          'mabs_nuv', 'mabs_r', 'mabs_k', 'mass_med'].copy()                                                                                    
                                                                                                                                                        
cut_catalog.rename_column('zfinal', 'z')                                                                                                                 
                                                                                                                                                        
# z_unc saved as 1 sigma uncertainty from 68% confidence interval half-width                                                                                                          
cut_catalog['z_unc'] = (cut_catalog['zpdf_u68'] - cut_catalog['zpdf_l68']) / 2 

In [ ]:
# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

# Sky coverage (assuming contiguous rectangular field)
ra_min, ra_max = cut_catalog['ra'].min(), cut_catalog['ra'].max()
dec_min, dec_max = cut_catalog['dec'].min(), cut_catalog['dec'].max()
dec_mean_rad = np.radians((dec_min + dec_max) / 2)
area_sqdeg = (ra_max - ra_min) * np.cos(dec_mean_rad) * (dec_max - dec_min)
print(f'Approximate sky coverage: {area_sqdeg:.2f} deg^2')

# object stats
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

# finally we save the cut catalog to a fits file
cut_catalog.write('../../data/catalogs/COSMOS2025_cut.fits', overwrite=False) 

In [ ]:
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(
    alerts=None,
    plot_catalogs=True,
    plot_ddf=False,
    catalog_path="../../data/catalogs",
    title="Catalog Coverage Skymap",
)

This code processes the CLAUDS catalogs from Subaru and GHST
- separate catalogs for Cosmos, Deep23, and XMMLss fields, fourth field is northern hemisphere
- They test HSCpipe as well as the commonly used Source Extractor+LePhare pipeline, we use the latter.

In [ ]:
# we load the full downloaded catalog
# download XMM-LSS photometry, DEEP2-3 photometry, E-COSMOS photometry

with fits.open('../../data/catalogs/raw_catalogs/XMMLSS_11bands-SExtractor-Lephare.fits') as hdul:
    hdul.info()

t = Table.read('../../data/catalogs/raw_catalogs/XMMLSS_11bands-SExtractor-Lephare.fits')                                                                                                 
t[0:5]

In [ ]:
# inspect catalog contents, ie histogram redshift values

import matplotlib.pyplot as plt

plot = plt.hist(t['ZPHOT'], bins=50, range=(0,6))

In [ ]:
# values to select on are specific to the catalog                                                                                                        
                                                                                                                                                        
# cuts for CLAUDS                                                                                
# lephare type: 0=galaxy, 1=star, 2=AGN                                                                                                                  
galaxies = t[
      (t['MASK'] == 0) &          # not in bright star mask
      (t['ST_TRAIL'] == 0) &       # not in satellite trail
      (t['OBJ_TYPE'] == 0) &       # galaxy (not star or QSO)
      (t['Z_BEST'] > 0) &          # valid photo-z
      (t['Z_BEST'] < 6) &
      (t['i'] < 25.0)              # reliable photo-z depth
]                    
                                                                                                                                                        
print(f'selected {len(galaxies)} from {len(t)} total objects')  

magerr = np.clip(galaxies['MAGERR_APER_2s_i'], 1e-6, None)                                                                                                                  
snr = 1.0857 / magerr                                                                                                                           
# RMS source size from ellipse axes (degrees -> arcsec)                                                                                                                     
sigma_ab_arcsec = np.sqrt(galaxies['A_WORLD'] * galaxies['B_WORLD']) * 3600                                                                                                                                                                 
# Statistical centroid uncertainty                                                                                                                                        
sigma_stat = sigma_ab_arcsec / snr                                                                                                                                                                                                                                                                                                            
sigma_sys = 0.010  # HSC astrometric systematic floor ~10 mas (Aihara+2018)                                                                                                                                                                                                                                                                                                                 
galaxies['pos_unc'] = np.sqrt(sigma_stat**2 + sigma_sys**2)

# z_unc saved as 1 sigma uncertainty from 68% confidence interval half-width                                                                                                          
galaxies['z_unc'] = (galaxies['Z_BEST68_HIGH'] - galaxies['Z_BEST68_LOW']) / 2 
                                                                                                                                                        
# standardize column names                                                                                                                               
cut_catalog = galaxies['ID', 'RA', 'DEC', 'Z_BEST', 'z_unc', 'pos_unc', 'Z_BEST68_LOW', 'Z_BEST68_HIGH',                                                                                              
                        'CHI_BEST', 'OBJ_TYPE', 'CLASS_STAR_HSC_I', 'i', 'i_err', 'MAG_APER_2s_i', 'MAGERR_APER_2s_i'].copy()                                                                                    

cut_catalog.rename_column('ID', 'id')                                                                                                                                     
cut_catalog.rename_column('RA', 'ra')
cut_catalog.rename_column('DEC', 'dec')
cut_catalog.rename_column('Z_BEST', 'z')                                                                                                                                                                                                                                                

In [ ]:
# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

# Sky coverage (assuming contiguous rectangular field)
ra_min, ra_max = cut_catalog['ra'].min(), cut_catalog['ra'].max()
dec_min, dec_max = cut_catalog['dec'].min(), cut_catalog['dec'].max()
dec_mean_rad = np.radians((dec_min + dec_max) / 2)
area_sqdeg = (ra_max - ra_min) * np.cos(dec_mean_rad) * (dec_max - dec_min)
print(f'Approximate sky coverage: {area_sqdeg:.2f} deg^2')

# object stats
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

# finally we save the cut catalog to a fits file
cut_catalog.write('../../data/catalogs/CLAUDS_SourceExtractor_XMMLSS_cut.fits', overwrite=False) 

In [ ]:
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(
    alerts=None,
    plot_catalogs=True,
    plot_ddf=False,
    catalog_path="../../data/catalogs",
    title="Catalog Coverage Skymap",
)

This code processes the VVDS, VUDS, VANDELS catalogs.

These are the specz catalogs we use to validate LBG photometric selection. They have a much smaller number of objects given that they are speczs, but we may as well add them to the crossmatch pipeline.

In [ ]:
# ── VUDS ──────────────────────────────────────────────────────────────────────                                                                                                         
df_vuds = pd.read_csv('../../data/specz_cats/cesam_vuds_spectra_dr1_ecdfs_catalog_1668100230.csv')
df_vuds = df_vuds[df_vuds['zflags'].isin([3, 4])]
df_vuds = df_vuds.rename(columns={'alpha': 'ra', 'delta': 'dec', 'z_spec': 'z', 'zflags': 'z_unc', 'id_iau': 'id'})
df_vuds['z_source'] = 'vuds'                                                                                                                                                             
df_vuds = df_vuds[['id', 'ra', 'dec', 'z', 'z_unc', 'z_source']]                                                                                                                         
print(f'VUDS:    {len(df_vuds)} objects after zflag cut')                                                                                                                                
                                                                                                                                                                                        
# ── VVDS ──────────────────────────────────────────────────────────────────────                                                                                                         
with gzip.open('../../data/specz_cats/cesam_vvds_spCDFS_DEEP.txt.gz', 'rt') as f:                                                                                                        
    for i, line in enumerate(f):                                                                                                                                                         
        if i == 8:                                                                                                                                                                       
            cols = line.strip().lstrip('# ').split()                                                                                                                                     
            break                                                                                                                                                                        
                                                                                                                                                                                        
df_vvds = pd.read_csv('../../data/specz_cats/cesam_vvds_spCDFS_DEEP.txt.gz',                                                                                                             
                        compression='gzip', sep=r'\s+', skiprows=9, names=cols)                                                                                                         
df_vvds = df_vvds.rename(columns={'ALPHA': 'ra', 'DELTA': 'dec', 'Z': 'z', 'ZFLAGS': 'z_unc', 'ID-IAU': 'id'})                                                                           
df_vvds = df_vvds[df_vvds['z_unc'].isin([3, 4])]                                                                                                                                         
df_vvds['z_source'] = 'vvds'                                                                                                                                                             
df_vvds = df_vvds[['id', 'ra', 'dec', 'z', 'z_unc', 'z_source']]                                                                                                                         
print(f'VVDS:    {len(df_vvds)} objects after zflag cut')                                                                                                                                
                                                                                                                                                                                        
# ── VANDELS ───────────────────────────────────────────────────────────────────                                                                                                         
df_vandels = Table.read('../../data/specz_cats/vandels_catalog.fits',                                                                                                                    
                        character_as_bytes=False).to_pandas()                                                                                                                            
df_vandels = df_vandels.rename(columns={'zspec': 'z', 'zflg': 'z_unc', 'id': 'id', 'alpha': 'ra', 'delta': 'dec'})                                                                       
df_vandels = df_vandels[df_vandels['z_unc'].isin([3, 4])]                                                                                                                                
df_vandels = df_vandels[df_vandels['z'].notna()]                                                                                                                                         
df_vandels['z_source'] = 'vandels'                                                                                                                                                       
df_vandels = df_vandels[['id', 'ra', 'dec', 'z', 'z_unc', 'z_source']]                                                                                                                   
print(f'VANDELS: {len(df_vandels)} objects after zflag cut') 

In [ ]:
# ── Combine ────────────────────────────────────────────────────────────                                                                                                         
cut_catalog = pd.concat([df_vuds, df_vvds, df_vandels], ignore_index=True)                                                                                                               
print(f'\nTotal: {len(cut_catalog)} objects')  

# placeholder z_unc = 0.01
cut_catalog['z_unc'] = 0.01

# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

# Sky coverage (assuming contiguous rectangular field)
ra_min, ra_max = cut_catalog['ra'].min(), cut_catalog['ra'].max()
dec_min, dec_max = cut_catalog['dec'].min(), cut_catalog['dec'].max()
dec_mean_rad = np.radians((dec_min + dec_max) / 2)
area_sqdeg = (ra_max - ra_min) * np.cos(dec_mean_rad) * (dec_max - dec_min)
print(f'Approximate sky coverage: {area_sqdeg:.2f} deg^2')

# object stats
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

In [ ]:
# ── save ────────────────────────────────────────────────────────────                                                                                                         
out_path = '../../data/catalogs/SPECZ_LBG_cats_cut.fits'                                                                                                                                 
Table.from_pandas(cut_catalog).write(out_path, overwrite=False)                                                                                                                          
print(f'\nSaved to {out_path}')